In [1]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape 
from pyvis.network import Network
import networkx as nx

In [2]:
with open('data/metro_stops.json') as f:
    metro_stops = json.load(f)

with open('data/metro_trajectories.json') as f:
    metro_trajectories = json.load(f)

excluded = ['FM','L10N','L10S','L11','L9N','L9S','L9NL10N','L9SL10S','TM']


In [3]:
df_stops = []
for stop in metro_stops['features']:
    properties = stop.get('properties', {})
    nom_estacio = properties.get('NOM_ESTACIO')
    df_stops.append({
            'name': nom_estacio,
            'geometry': shape(stop['geometry'])
        })

df_stops = pd.DataFrame(df_stops)
geo_df_stops = gpd.GeoDataFrame(df_stops)
geo_df_stops

,name,geometry
0,Hospital de Bellvitge,POINT (2.10724 41.34468)
1,Bellvitge,POINT (2.11092 41.35097)
2,Av. Carrilet,POINT (2.10265 41.35852)
3,Rambla Just Oliveras,POINT (2.09975 41.36409)
4,Can Serra,POINT (2.10275 41.36769)
...,...,...
135,Torre Baró | Vallbona,POINT (2.17988 41.4592)
136,Ciutat Meridiana,POINT (2.17465 41.46081)
137,Can Cuiàs,POINT (2.17306 41.46241)
138,Telefèric Parc de Montjuïc,POINT (2.16327 41.36896)


In [4]:
df_trajectories_all = []
for trajectory in metro_trajectories['features']:
    properties = trajectory.get('properties', {})
    tram = properties.get('NOM_TRAM_LINIA')
    nom_linia = properties.get('NOM_LINIA')
    df_trajectories_all.append({
            'tram': tram,
            'nom_inici': properties.get('NOM_ESTACIO_INI'),
            'nom_fi': properties.get('NOM_ESTACIO_FI'),
            'linia': nom_linia,
            'geometry': shape(trajectory['geometry'])
        })
df_trajectories_all = pd.DataFrame(df_trajectories_all)
geo_df_trajectories_all = gpd.GeoDataFrame(df_trajectories_all, geometry='geometry')
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Inici') == False]
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Final') == False]
geo_df_trajectories_all = geo_df_trajectories_all[~geo_df_trajectories_all['linia'].isin(excluded)]
geo_df_trajectories_all = geo_df_trajectories_all.set_crs(4326)
geo_df_trajectories_all['distance'] = (geo_df_trajectories_all.to_crs(25831).geometry.length)
geo_df_trajectories_all

,tram,nom_inici,nom_fi,linia,geometry,distance
1,Hospital de Bellvitge - Bellvitge,Hospital de Bellvitge,Bellvitge,L1,"MULTILINESTRING ((2.10724 41.34468, 2.10773 41...",882.209743
2,Bellvitge - Av. Carrilet,Bellvitge,Av. Carrilet,L1,"MULTILINESTRING ((2.11092 41.35098, 2.11052 41...",1133.654863
3,Av. Carrilet - Rambla Just Oliveras,Av. Carrilet,Rambla Just Oliveras,L1,"MULTILINESTRING ((2.10263 41.35855, 2.10202 41...",660.855917
4,Rambla Just Oliveras - Can Serra,Rambla Just Oliveras,Can Serra,L1,"MULTILINESTRING ((2.09975 41.36409, 2.09946 41...",572.330154
5,Can Serra - Florida,Can Serra,Florida,L1,"MULTILINESTRING ((2.10276 41.36769, 2.10381 41...",616.671198
...,...,...,...,...,...,...
122,Virrei Amat - Vilapicina,Virrei Amat,Vilapicina,L5,"MULTILINESTRING ((2.17491 41.4297, 2.17486 41....",636.238350
123,Vilapicina - Horta,Vilapicina,Horta,L5,"MULTILINESTRING ((2.16762 41.43047, 2.16564 41...",683.078591
124,Horta - El Carmel,Horta,El Carmel,L5,"MULTILINESTRING ((2.15997 41.42969, 2.15974 41...",851.210213
125,El Carmel - El Coll | La Teixonera,El Carmel,El Coll | La Teixonera,L5,"MULTILINESTRING ((2.1551 41.42445, 2.15513 41....",838.886751


In [5]:
joined = geo_df_trajectories_all.merge(geo_df_stops, left_on='nom_inici', right_on='name', how='left')
joined.rename(columns={'geometry_x': 'trajectory_geometry', 'geometry_y': 'starting_geometry'}, inplace=True)
joined = joined.merge(geo_df_stops, left_on='nom_fi', right_on='name', how='left')
joined.rename(columns={'geometry': 'ending_geometry'}, inplace=True)
joined.drop(columns=['name_x', 'name_y',], inplace=True)
joined

,tram,nom_inici,nom_fi,linia,trajectory_geometry,distance,starting_geometry,ending_geometry
0,Hospital de Bellvitge - Bellvitge,Hospital de Bellvitge,Bellvitge,L1,"MULTILINESTRING ((2.10724 41.34468, 2.10773 41...",882.209743,POINT (2.10724 41.34468),POINT (2.11092 41.35097)
1,Bellvitge - Av. Carrilet,Bellvitge,Av. Carrilet,L1,"MULTILINESTRING ((2.11092 41.35098, 2.11052 41...",1133.654863,POINT (2.11092 41.35097),POINT (2.10265 41.35852)
2,Av. Carrilet - Rambla Just Oliveras,Av. Carrilet,Rambla Just Oliveras,L1,"MULTILINESTRING ((2.10263 41.35855, 2.10202 41...",660.855917,POINT (2.10265 41.35852),POINT (2.09975 41.36409)
3,Rambla Just Oliveras - Can Serra,Rambla Just Oliveras,Can Serra,L1,"MULTILINESTRING ((2.09975 41.36409, 2.09946 41...",572.330154,POINT (2.09975 41.36409),POINT (2.10275 41.36769)
4,Can Serra - Florida,Can Serra,Florida,L1,"MULTILINESTRING ((2.10276 41.36769, 2.10381 41...",616.671198,POINT (2.10275 41.36769),POINT (2.11003 41.36832)
...,...,...,...,...,...,...,...,...
113,Virrei Amat - Vilapicina,Virrei Amat,Vilapicina,L5,"MULTILINESTRING ((2.17491 41.4297, 2.17486 41....",636.238350,POINT (2.17491 41.4297),POINT (2.16762 41.43047)
114,Vilapicina - Horta,Vilapicina,Horta,L5,"MULTILINESTRING ((2.16762 41.43047, 2.16564 41...",683.078591,POINT (2.16762 41.43047),POINT (2.15997 41.42969)
115,Horta - El Carmel,Horta,El Carmel,L5,"MULTILINESTRING ((2.15997 41.42969, 2.15974 41...",851.210213,POINT (2.15997 41.42969),POINT (2.1551 41.42445)
116,El Carmel - El Coll | La Teixonera,El Carmel,El Coll | La Teixonera,L5,"MULTILINESTRING ((2.1551 41.42445, 2.15513 41....",838.886751,POINT (2.1551 41.42445),POINT (2.14835 41.422)


In [6]:
point = shape({
    'type': 'Point',
    'coordinates':  (2.170227352668114, 41.399696457926865)
})

geo_df_stops = geo_df_stops.set_crs(4326)
point_gdf = gpd.GeoSeries([point], crs=4326).to_crs(25831)
point_metric = point_gdf.iloc[0]

geo_df_metric = geo_df_stops.to_crs(25831)

geo_df_stops['nearby'] = (geo_df_metric.geometry.distance(point_metric) <= 200)
geo_df_stops['distance_to_point'] = geo_df_metric.geometry.distance(point_metric)
closest_stop_start = geo_df_stops[geo_df_stops['distance_to_point'] == geo_df_stops['distance_to_point'].min()]
closest_stop_start


,name,geometry,nearby,distance_to_point
79,Verdaguer,POINT (2.16831 41.39997),True,162.867278


In [7]:
destination = shape({
    'type': 'Point',
    'coordinates': (2.1130282510923517, 41.38913433225749)
})

destination_gdf = gpd.GeoSeries([destination], crs=4326).to_crs(25831)

geo_df_stops['distance_to_destination'] = geo_df_metric.geometry.distance(destination_gdf.iloc[0])
geo_df_stops['closest_stop_end'] = (geo_df_stops['distance_to_destination']== geo_df_stops['distance_to_destination'].min())
geo_df_stops[geo_df_stops['closest_stop_end'] | geo_df_stops['nearby']]
closest_stop_end = geo_df_stops[geo_df_stops['distance_to_destination'] == geo_df_stops['distance_to_destination'].min()]
closest_stop_end

,name,geometry,nearby,distance_to_point,distance_to_destination,closest_stop_end
47,Palau Reial,POINT (2.11854 41.38598),False,4581.966598,578.692498,True


In [8]:
line_colors = {
    'L1': 'red',
    'L2': 'purple',
    'L3': 'green',
    'L4': 'yellow',
    'L5': 'blue',
}

In [9]:
from pyvis.network import Network
import networkx as nx
from IPython.display import IFrame

G = nx.MultiGraph()
stations = set(joined['nom_inici']).union(set(joined['nom_fi']))

for station in stations:
    G.add_node(station,label=station,title=station,size = 200,color='steelblue')

for _, row in joined.iterrows():
    color = line_colors.get(row['linia'], 'gray')
    G.add_edge(row['nom_inici'],row['nom_fi'],weight=100,color=color,title=f"{row['linia']} - {row['distance']:.0f}m",)


# Create network
nt = Network(
    height='700px',
    width='100%',
    notebook=True
)

G.add_node('Start', label='Start', title='Starting Point', size=200, color='green')
G.add_edge('Start', closest_stop_start.iloc[0]['name'], weight=100, color='black', title='Walk to Closest Stop')

G.add_node('End', label='End', title='Destination', size=200, color='blue')
G.add_edge(closest_stop_end.iloc[0]['name'], 'End', weight=100, color='black', title='Walk to Destination')

nt.from_nx(G)
nt.barnes_hut()
nt.show('metro_network.html')
IFrame('metro_network.html', width=1000, height=700)

metro_network.html
